# Module 07: Order Reviews EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_order_reviews_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

reviews = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {reviews.shape}")
reviews.head(3)

Loaded raw dataset shape: (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [2]:
audit_df = pd.DataFrame(
    {
        "Data Type": reviews.dtypes,
        "Missing Values": reviews.isna().sum(),
        "Missing %": (reviews.isna().mean() * 100).round(2),
        "Unique Values": reviews.nunique(),
    }
)
display(audit_df)

,Data Type,Missing Values,Missing %,Unique Values
review_id,object,0,0.00,98410
order_id,object,0,0.00,98673
review_score,int64,0,0.00,5
review_comment_title,object,87656,88.34,4527
review_comment_message,object,58247,58.70,36159
review_creation_date,object,0,0.00,636
review_answer_timestamp,object,0,0.00,98248


### 3. Data Type Optimization & Duplicate Removal

To transition our dataset from the Bronze layer to the Silver layer, we remove exact duplicate records and enforce robust data types:

* **Identifiers & Text (`string`):** Standardize `review_id`, `order_id`, `review_comment_title`, and `review_comment_message` into explicit string types, preserving `NaN` values for optional text fields without artificial placeholders.
* **Numeric Score (`int64`):** Enforce strict integer formatting for `review_score`.
* **Timestamps (`datetime`):** Convert `review_creation_date` and `review_answer_timestamp` into proper datetime objects for accurate temporal analysis.

In [3]:
initial_len = len(reviews)
reviews_clean = reviews.drop_duplicates().copy()
print(f"Dropped duplicate rows: {initial_len - len(reviews_clean):,}")

reviews_clean["review_id"] = reviews_clean["review_id"].astype("string")
reviews_clean["order_id"] = reviews_clean["order_id"].astype("string")
reviews_clean["review_score"] = reviews_clean["review_score"].astype("int64")
reviews_clean["review_comment_title"] = reviews_clean["review_comment_title"].astype("string")
reviews_clean["review_comment_message"] = reviews_clean["review_comment_message"].astype("string")
reviews_clean["review_creation_date"] = pd.to_datetime(reviews_clean["review_creation_date"])
reviews_clean["review_answer_timestamp"] = pd.to_datetime(reviews_clean["review_answer_timestamp"])

display(reviews_clean.dtypes.to_frame(name="Silver Layer Data Type"))

Dropped duplicate rows: 0


,Silver Layer Data Type
review_id,string[python]
order_id,string[python]
review_score,int64
review_comment_title,string[python]
review_comment_message,string[python]
review_creation_date,datetime64[ns]
review_answer_timestamp,datetime64[ns]


### 4. Data Integrity Checks & Anomaly Detection

To ensure temporal consistency and catch logical errors in the reviews dataset, we validate date boundaries and time sequence logic:

* **Temporal Bounds Check:** Ensuring review creation and answer timestamps fall within the historical operational window (2016–2020).
* **Logical Sequence Check:** Verifying that the review answer timestamp is strictly greater than or equal to the review creation date.

In [4]:
audit_checks = {
    "Review answer earlier than creation":
        reviews_clean["review_answer_timestamp"] < reviews_clean["review_creation_date"],
    "Review creation date out of historical bounds":
        (reviews_clean["review_creation_date"] < "2016-01-01") | (reviews_clean["review_creation_date"] > "2020-01-01"),
    "Review answer date out of historical bounds":
        (reviews_clean["review_answer_timestamp"] < "2016-01-01") | (reviews_clean["review_answer_timestamp"] > "2020-01-01")
}

for label, condition in audit_checks.items():
    print(f"{label}: {condition.sum()}")

Review answer earlier than creation: 0
Review creation date out of historical bounds: 0
Review answer date out of historical bounds: 0


### 5. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [5]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {reviews.shape}")
print(f"Cleaned shape (Silver): {reviews_clean.shape}")
print(f"Total rows dropped: {len(reviews) - len(reviews_clean):,}")

dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': reviews.dtypes,
    'Silver (Clean)': reviews_clean.dtypes
})

print("\nData Types Transformation:")
display(dtypes_comparison)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (99224, 7)
Cleaned shape (Silver): (99224, 7)
Total rows dropped: 0

Data Types Transformation:


,Bronze (Raw),Silver (Clean)
review_id,object,string[python]
order_id,object,string[python]
review_score,int64,int64
review_comment_title,object,string[python]
review_comment_message,object,string[python]
review_creation_date,object,datetime64[ns]
review_answer_timestamp,object,datetime64[ns]


## 6. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks tailored to review integrity and temporal logic:

* **Mandatory Identifiers & Attributes:** Ensure core fields (`review_id`, `order_id`, `review_score`, `review_creation_date`, `review_answer_timestamp`) contain no missing values.
* **Score Domain Validity:** Confirm that `review_score` values strictly fall within the valid scale range (1 to 5).
* **Temporal Sequence Logic:** Verify that review answer timestamps do not precede review creation dates.
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [6]:
critical_cols = ['review_id', 'order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']
for col in critical_cols:
    assert reviews_clean[col].notna().all(), f"Assertion failed: NULL values found in mandatory column '{col}'"

valid_scores = reviews_clean['review_score'].isin([1, 2, 3, 4, 5])
assert valid_scores.all(), "Assertion failed: Invalid review score values found outside the 1-5 range"

valid_sequence = reviews_clean['review_answer_timestamp'] >= reviews_clean['review_creation_date']
assert valid_sequence.all(), "Assertion failed: Review answer timestamp is earlier than creation date"

assert len(reviews_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
reviews_clean.to_parquet("../data/reviews_clean.parquet", index=False)
print("Reviews dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.
Reviews dataset successfully saved to Parquet.
